# LLaMA-2-7B fake BFP + fused OB-Skip

Evaluate LLaMA-2-7B on the complete WikiText-2 test split using the verified fused Triton OB-Skip kernel and the same fake W/A BFP quantization as `bfp.ipynb`.

Fixed configuration: **BFP8 (1S7M), shared E5, Group 32**. One run sweeps **T=8, 9, 10, 11, 12**. Fake-quantized weights and activations remain FP16; Group partials and accepted accumulation use FP32; Linear output is FP16.

Formal evaluation settings: **non-overlapping 2048-token blocks; the incomplete final block is dropped**.

In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece tqdm

In [ ]:
import json
import os
import platform
import time
import zipfile
from dataclasses import asdict, dataclass
from getpass import getpass
from pathlib import Path

import datasets
import torch
import triton
import triton.language as tl
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-2-7b-hf"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = "non_overlapping_2048_drop_remainder"
FP16_BASELINE_PPL = 5.472103118896484
BFP_REFERENCE_PPL_BY_MANTISSA = {7: 5.477353096008301}
THRESHOLD_SWEEP = tuple(range(8, 13))
if not THRESHOLD_SWEEP:
    raise ValueError("THRESHOLD_SWEEP must not be empty.")
if THRESHOLD_SWEEP != tuple(range(THRESHOLD_SWEEP[0], THRESHOLD_SWEEP[-1] + 1)):
    raise ValueError("THRESHOLD_SWEEP must be a contiguous ascending range.")
THRESHOLD_RANGE_TAG = f"t{THRESHOLD_SWEEP[0]}-t{THRESHOLD_SWEEP[-1]}"

@dataclass(frozen=True)
class BFPConfig:
    block_size: int = 16
    shared_exponent_bits: int = 5
    mantissa_bits: int = 3  # Excludes sign: 1S3M = BFP4.
    rounding: str = "nearest"
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 2048
    quantize_lm_head: bool = False

    def validate(self):
        if self.block_size < 16 or self.block_size & (self.block_size - 1):
            raise ValueError("block_size must be a power of two >= 16.")
        if self.shared_exponent_bits < 2:
            raise ValueError("shared_exponent_bits must be at least 2.")
        if self.mantissa_bits <= 0:
            raise ValueError("mantissa_bits must be positive.")
        if self.rounding not in {"nearest", "trunc"}:
            raise ValueError("rounding must be 'nearest' or 'trunc'.")

@dataclass(frozen=True)
class OBSkipConfig:
    threshold_bits: int = 12
    enabled: bool = True
    max_eval_tokens: int | None = None

    def validate(self):
        if self.threshold_bits <= 0:
            raise ValueError("threshold_bits must be positive.")
        if self.max_eval_tokens is not None:
            if self.max_eval_tokens < CONTEXT_LENGTH or self.max_eval_tokens % CONTEXT_LENGTH != 0:
                raise ValueError("max_eval_tokens must be None or a positive multiple of CONTEXT_LENGTH.")

@dataclass(frozen=True)
class TritonKernelConfig:
    block_m: int = 32
    block_n: int = 64
    group_k: int = 16
    num_warps: int = 4
    num_stages: int = 2

    def validate(self):
        for name in ("block_m", "block_n", "group_k"):
            value = getattr(self, name)
            if value <= 0 or value & (value - 1):
                raise ValueError(f"{name} must be a positive power of two.")
        if self.group_k < 16:
            raise ValueError("The fused kernel requires group_k >= 16 (tl.dot minimum).")


BFP = BFPConfig()
OB_SKIP_TEMPLATE = OBSkipConfig(threshold_bits=THRESHOLD_SWEEP[0])
KERNEL = TritonKernelConfig()
BFP.validate()
OB_SKIP_TEMPLATE.validate()
KERNEL.validate()
if KERNEL.group_k != BFP.block_size:
    raise ValueError(
        f"group_k ({KERNEL.group_k}) must equal BFP block_size ({BFP.block_size}); "
        "otherwise the fused kernel forms partial dot products at a different "
        "granularity than the BFP blocks and every DEWA decision is taken on the "
        "wrong operand."
    )

BFP_BITS = 1 + BFP.mantissa_bits
OUTPUT_DIR = Path(f"obskip-bfp{BFP_BITS}-g{BFP.block_size}-{THRESHOLD_RANGE_TAG}-s2048-results")
ARCHIVE_PATH = Path(f"llama2-7b-obskip-bfp{BFP_BITS}-g{BFP.block_size}-{THRESHOLD_RANGE_TAG}-s2048.zip")

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires an NVIDIA CUDA GPU.")

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False
print(BFP)
print(OB_SKIP_TEMPLATE)
print(KERNEL)
print(f"Triton: {triton.__version__}")
print(f"Threshold sweep: {THRESHOLD_SWEEP}")
print(f"Output directory: {OUTPUT_DIR}")

## BFP quantization

Weights and activations use the same BFP convention as `bfp.ipynb`: contiguous blocks along the last dimension, signed shared exponent, symmetric sign-magnitude mantissa range, and dequantization back to FP16 before Linear arithmetic.

In [ ]:
def _quantize_bfp_rows(rows, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size

    if padding:
        flat = F.pad(flat, (0, padding))

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    shared_exp = torch.floor(torch.log2(safe_max))

    exp_min = -(1 << (config.shared_exponent_bits - 1))
    exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
    shared_exp = shared_exp.clamp(exp_min, exp_max)
    shared_exp = torch.where(max_abs == 0, torch.zeros_like(shared_exp), shared_exp)

    step = torch.pow(2.0, shared_exp - (config.mantissa_bits - 1))
    mantissa = blocks / step
    mantissa = torch.round(mantissa) if config.rounding == "nearest" else torch.trunc(mantissa)
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = mantissa.clamp(-mantissa_max, mantissa_max)

    dequantized = (mantissa * step).reshape(flat.size(0), padded_width)
    dequantized = dequantized[:, :width].reshape(original_shape)
    return dequantized.to(rows.dtype)


def quantize_bfp(tensor, config, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)

    if flat.size(0) <= chunk_rows:
        return _quantize_bfp_rows(tensor, config)

    output = torch.empty_like(flat)
    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        output[start:end] = _quantize_bfp_rows(flat[start:end], config)

    return output.reshape_as(tensor)


@torch.no_grad()
def quantize_weight_in_place(weight, config):
    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        weight[start:end].copy_(_quantize_bfp_rows(weight[start:end], config))

## Fused functional OB-Skip accumulator

A 2D Triton launch assigns one program to each output tile. Inside that single kernel, every program walks through K in Group-32 steps, computes an FP32 partial dot product from the fake-BFP FP16 tensors, applies the exponent-gap rule, and keeps the accepted accumulator in FP32.

Let `delta = E_new - E_old`: discard New when `delta <= -T`, replace Old when `delta >= T`, and otherwise add in FP32. The first compatible implementation deliberately uses direct `floor(log2(abs(x)))` exponent extraction and explicit reductions; lower-level bit extraction and launch reordering can be optimized after parity is confirmed.

In [ ]:
STAT_NAMES = (
    "initial_or_zero_old_load",
    "zero_new",
    "skip_new",
    "replace_old",
    "normal_add",
    "total_nonzero_decisions",
)


def _binary_exponent(value):
    tiny = torch.finfo(torch.float32).tiny
    return torch.floor(torch.log2(value.abs().clamp_min(tiny)))


def ob_skip_update(old, new, threshold_bits, enabled=True):
    old_nonzero = old != 0
    new_nonzero = new != 0
    zero_new = ~new_nonzero
    load_new = ~old_nonzero & new_nonzero
    both_nonzero = old_nonzero & new_nonzero

    if enabled:
        delta_exp = _binary_exponent(new) - _binary_exponent(old)
        skip_new = both_nonzero & (delta_exp <= -threshold_bits)
        replace_old = both_nonzero & (delta_exp >= threshold_bits)
        normal_add = both_nonzero & ~(skip_new | replace_old)
    else:
        skip_new = torch.zeros_like(both_nonzero)
        replace_old = torch.zeros_like(both_nonzero)
        normal_add = both_nonzero

    updated = old
    updated = torch.where(load_new | replace_old, new, updated)
    updated = torch.where(normal_add, old + new, updated)

    counts = torch.stack((
        load_new.sum(),
        zero_new.sum(),
        skip_new.sum(),
        replace_old.sum(),
        normal_add.sum(),
        both_nonzero.sum(),
    )).to(torch.int64)
    return updated, counts


@triton.jit
def _ob_skip_linear_kernel(
    x_ptr,
    weight_ptr,
    bias_ptr,
    output_ptr,
    stats_ptr,
    M,
    N,
    K,
    stride_xm,
    stride_xk,
    stride_wn,
    stride_wk,
    stride_om,
    stride_on,
    THRESHOLD_BITS: tl.constexpr,
    ENABLED: tl.constexpr,
    HAS_BIAS: tl.constexpr,
    COLLECT_STATS: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    GROUP_K: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, GROUP_K)
    valid_output = (offs_m[:, None] < M) & (offs_n[None, :] < N)

    x_ptrs = x_ptr + offs_m[:, None] * stride_xm + offs_k[None, :] * stride_xk
    weight_ptrs = (
        weight_ptr
        + offs_n[:, None] * stride_wn
        + offs_k[None, :] * stride_wk
    )

    old = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    load_count = tl.zeros((1,), dtype=tl.int32)
    zero_new_count = tl.zeros((1,), dtype=tl.int32)
    skip_new_count = tl.zeros((1,), dtype=tl.int32)
    replace_old_count = tl.zeros((1,), dtype=tl.int32)
    normal_add_count = tl.zeros((1,), dtype=tl.int32)
    decision_count = tl.zeros((1,), dtype=tl.int32)

    for _ in range(0, tl.cdiv(K, GROUP_K)):
        x_block = tl.load(
            x_ptrs,
            mask=offs_m[:, None] < M,
            other=0.0,
        )
        weight_block = tl.load(
            weight_ptrs,
            mask=offs_n[:, None] < N,
            other=0.0,
        )
        new = tl.dot(x_block, tl.trans(weight_block)).to(tl.float32)

        old_nonzero = old != 0.0
        new_nonzero = new != 0.0
        load_new = valid_output & ~old_nonzero & new_nonzero
        zero_new = valid_output & ~new_nonzero
        both_nonzero = valid_output & old_nonzero & new_nonzero

        if ENABLED:
            tiny = 1.1754943508222875e-38
            old_exp = tl.floor(tl.log2(tl.maximum(tl.abs(old), tiny)))
            new_exp = tl.floor(tl.log2(tl.maximum(tl.abs(new), tiny)))
            delta_exp = new_exp - old_exp
            skip_new = both_nonzero & (delta_exp <= -THRESHOLD_BITS)
            replace_old = both_nonzero & (delta_exp >= THRESHOLD_BITS)
            normal_add = both_nonzero & ~(skip_new | replace_old)
        else:
            skip_new = both_nonzero & False
            replace_old = both_nonzero & False
            normal_add = both_nonzero

        updated = tl.where(load_new | replace_old, new, old)
        old = tl.where(normal_add, old + new, updated)

        if COLLECT_STATS:
            load_count += tl.sum(
                tl.sum(load_new.to(tl.int32), axis=1),
                axis=0,
            )
            zero_new_count += tl.sum(
                tl.sum(zero_new.to(tl.int32), axis=1),
                axis=0,
            )
            skip_new_count += tl.sum(
                tl.sum(skip_new.to(tl.int32), axis=1),
                axis=0,
            )
            replace_old_count += tl.sum(
                tl.sum(replace_old.to(tl.int32), axis=1),
                axis=0,
            )
            normal_add_count += tl.sum(
                tl.sum(normal_add.to(tl.int32), axis=1),
                axis=0,
            )
            decision_count += tl.sum(
                tl.sum(both_nonzero.to(tl.int32), axis=1),
                axis=0,
            )

        x_ptrs += GROUP_K * stride_xk
        weight_ptrs += GROUP_K * stride_wk

    if HAS_BIAS:
        bias = tl.load(bias_ptr + offs_n, mask=offs_n < N, other=0.0)
        old = old + bias[None, :]

    output_ptrs = (
        output_ptr
        + offs_m[:, None] * stride_om
        + offs_n[None, :] * stride_on
    )
    tl.store(output_ptrs, old, mask=valid_output)

    if COLLECT_STATS:
        tl.atomic_add(
            stats_ptr + 0,
            tl.sum(load_count, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 1,
            tl.sum(zero_new_count, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 2,
            tl.sum(skip_new_count, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 3,
            tl.sum(replace_old_count, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 4,
            tl.sum(normal_add_count, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 5,
            tl.sum(decision_count, axis=0).to(tl.int64),
        )


def fused_ob_skip_linear(
    x,
    weight,
    bias,
    threshold_bits,
    enabled,
    stats,
    kernel_config,
):
    if x.dtype != torch.float16 or weight.dtype != torch.float16:
        raise TypeError("The fused kernel expects fake-BFP tensors stored as FP16.")

    in_features = x.shape[-1]
    if in_features != weight.shape[1]:
        raise ValueError("Input and weight dimensions do not match.")
    if in_features % kernel_config.group_k != 0:
        raise ValueError(
            f"in_features must be divisible by Group {kernel_config.group_k}."
        )

    original_shape = x.shape[:-1]
    x_flat = x.reshape(-1, in_features).contiguous()
    weight = weight.contiguous()
    M, K = x_flat.shape
    N = weight.shape[0]
    output = torch.empty((M, N), dtype=torch.float16, device=x.device)
    bias_arg = bias if bias is not None else weight

    grid = (
        triton.cdiv(M, kernel_config.block_m),
        triton.cdiv(N, kernel_config.block_n),
    )
    _ob_skip_linear_kernel[grid](
        x_flat,
        weight,
        bias_arg,
        output,
        stats,
        M,
        N,
        K,
        x_flat.stride(0),
        x_flat.stride(1),
        weight.stride(0),
        weight.stride(1),
        output.stride(0),
        output.stride(1),
        THRESHOLD_BITS=threshold_bits,
        ENABLED=enabled,
        HAS_BIAS=bias is not None,
        COLLECT_STATS=True,
        BLOCK_M=kernel_config.block_m,
        BLOCK_N=kernel_config.block_n,
        GROUP_K=kernel_config.group_k,
        num_warps=kernel_config.num_warps,
        num_stages=kernel_config.num_stages,
    )
    return output.reshape(*original_shape, N)


class FusedOBSkipLinear(nn.Module):
    def __init__(self, linear, bfp_config, ob_skip_config, kernel_config, layer_name):
        super().__init__()
        self.linear = linear
        self.bfp_config = bfp_config
        self.ob_skip_config = ob_skip_config
        self.kernel_config = kernel_config
        self.layer_name = layer_name
        self.register_buffer(
            "_ob_skip_counts",
            torch.zeros(len(STAT_NAMES), dtype=torch.int64, device=linear.weight.device),
            persistent=False,
        )

    def reset_stats(self):
        self._ob_skip_counts.zero_()

    def stats(self):
        values = self._ob_skip_counts.detach().cpu().tolist()
        return {name: int(value) for name, value in zip(STAT_NAMES, values)}

    def forward(self, x):
        x_bfp = quantize_bfp(x, self.bfp_config, self.bfp_config.activation_chunk_rows)
        return fused_ob_skip_linear(
            x_bfp,
            self.linear.weight,
            self.linear.bias,
            self.ob_skip_config.threshold_bits,
            self.ob_skip_config.enabled,
            self._ob_skip_counts,
            self.kernel_config,
        )


def replace_linear_layers(module, bfp_config, ob_skip_config, kernel_config, prefix=""):
    replaced = []

    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name
        if isinstance(child, nn.Linear):
            if full_name == "lm_head" and not bfp_config.quantize_lm_head:
                continue
            quantize_weight_in_place(child.weight, bfp_config)
            setattr(
                module,
                name,
                FusedOBSkipLinear(
                    child,
                    bfp_config,
                    ob_skip_config,
                    kernel_config,
                    full_name,
                ),
            )
            replaced.append(full_name)
        else:
            replaced.extend(
                replace_linear_layers(
                    child,
                    bfp_config,
                    ob_skip_config,
                    kernel_config,
                    full_name,
                )
            )

    return replaced


def _rates(stats):
    decisions = stats["total_nonzero_decisions"]
    if decisions == 0:
        return {
            "ob_skip_rate": 0.0,
            "new_discard_rate": 0.0,
            "old_replace_rate": 0.0,
            "normal_add_rate": 0.0,
        }
    return {
        "ob_skip_rate": (stats["skip_new"] + stats["replace_old"]) / decisions,
        "new_discard_rate": stats["skip_new"] / decisions,
        "old_replace_rate": stats["replace_old"] / decisions,
        "normal_add_rate": stats["normal_add"] / decisions,
    }


def collect_ob_skip_stats(model):
    per_layer = {}
    aggregate = {name: 0 for name in STAT_NAMES}

    for module in model.modules():
        if not isinstance(module, FusedOBSkipLinear):
            continue
        layer_stats = module.stats()
        layer_stats.update(_rates(layer_stats))
        per_layer[module.layer_name] = layer_stats
        for name in STAT_NAMES:
            aggregate[name] += layer_stats[name]

    aggregate.update(_rates(aggregate))
    return aggregate, per_layer


def configure_ob_skip(model, ob_skip_config, reset_stats=True):
    ob_skip_config.validate()
    configured_layers = 0

    for module in model.modules():
        if not isinstance(module, FusedOBSkipLinear):
            continue
        module.ob_skip_config = ob_skip_config
        if reset_stats:
            module.reset_stats()
        configured_layers += 1

    return configured_layers

## Reference parity tests

The tests compare the fused Triton result and all six counters against the original PyTorch Group-32 implementation before loading LLaMA-2-7B.

In [ ]:
device = torch.device("cuda")


def check_update(old_value, new_value, expected, threshold=12):
    old = torch.tensor([old_value], dtype=torch.float32, device=device)
    new = torch.tensor([new_value], dtype=torch.float32, device=device)
    actual, _ = ob_skip_update(old, new, threshold, enabled=True)
    torch.testing.assert_close(actual, torch.tensor([expected], device=device))


for threshold in THRESHOLD_SWEEP:
    check_update(1.0, 2.0**-threshold, 1.0, threshold)
    check_update(2.0**-threshold, 1.0, 1.0, threshold)
    check_update(1.0, 2.0**-(threshold - 1), 1.0 + 2.0**-(threshold - 1), threshold)
    check_update(2.0**-(threshold - 1), 1.0, 1.0 + 2.0**-(threshold - 1), threshold)
check_update(0.0, 1.0, 1.0)
check_update(1.0, 0.0, 1.0)
check_update(1.0, -1.0, 0.0)


def reference_group_linear(x, weight, threshold_bits, enabled, group_size):
    M, K = x.shape
    old = torch.zeros((M, weight.size(0)), dtype=torch.float32, device=x.device)
    counts = torch.zeros(len(STAT_NAMES), dtype=torch.int64, device=x.device)

    for start in range(0, K, group_size):
        new = F.linear(
            x[:, start:start + group_size].float(),
            weight[:, start:start + group_size].float(),
        )
        old, group_counts = ob_skip_update(
            old,
            new,
            threshold_bits,
            enabled,
        )
        counts += group_counts

    return old.to(torch.float16), counts


torch.manual_seed(1)
test_linear = nn.Linear(64, 37, bias=False, device=device, dtype=torch.float16)
test_x = torch.randn(5, 64, device=device, dtype=torch.float16)
quantize_weight_in_place(test_linear.weight, BFP)
test_x_bfp = quantize_bfp(test_x, BFP, BFP.activation_chunk_rows)

parity_cases = [(True, threshold) for threshold in THRESHOLD_SWEEP] + [(False, 12)]
for enabled, threshold in parity_cases:
    reference_output, reference_counts = reference_group_linear(
        test_x_bfp,
        test_linear.weight,
        threshold,
        enabled,
        BFP.block_size,
    )
    fused_counts = torch.zeros(len(STAT_NAMES), dtype=torch.int64, device=device)
    fused_output = fused_ob_skip_linear(
        test_x_bfp,
        test_linear.weight,
        None,
        threshold,
        enabled,
        fused_counts,
        KERNEL,
    )
    torch.cuda.synchronize()

    torch.testing.assert_close(fused_output, reference_output, rtol=0, atol=0)
    torch.testing.assert_close(fused_counts, reference_counts, rtol=0, atol=0)

print(f"Triton/PyTorch output and counter parity tests passed for {THRESHOLD_RANGE_TAG}.")

## Load model and apply fake BFP + fused OB-Skip

Accept the LLaMA-2 license and authenticate with a Colab secret named `HF_TOKEN`. The FP16 model is loaded once and its weights are fake-quantized to the fixed BFP configuration once. The threshold sweep reuses that model, changes only `T`, and resets all OB-Skip counters before each evaluation.

In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = getpass("HF_TOKEN: ")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map=0,
    attn_implementation="eager",
    token=token,
)
model.eval()
model.config.use_cache = False

quantized_layers = replace_linear_layers(model, BFP, OB_SKIP_TEMPLATE, KERNEL)
torch.cuda.empty_cache()

parameter_dtypes = {p.dtype for p in model.parameters() if p.is_floating_point()}
parameter_devices = {p.device.type for p in model.parameters()}
assert parameter_dtypes == {torch.float16}, parameter_dtypes
assert parameter_devices == {"cuda"}, parameter_devices
expected_quantized_layers = (
    model.config.num_hidden_layers * 7 + (1 if BFP.quantize_lm_head else 0)
)
assert len(quantized_layers) == expected_quantized_layers, (
    len(quantized_layers),
    expected_quantized_layers,
)

print(f"Quantized Linear layers: {len(quantized_layers)}")
print(f"First: {quantized_layers[0]}")
print(f"Last:  {quantized_layers[-1]}")

In [ ]:
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = "\n\n".join(dataset["text"])
input_ids = tokenizer(text, return_tensors="pt").input_ids

if OB_SKIP_TEMPLATE.max_eval_tokens is not None:
    input_ids = input_ids[:, :OB_SKIP_TEMPLATE.max_eval_tokens]

print(f"WikiText-2 {SPLIT} tokens: {input_ids.numel():,}")

In [ ]:
@torch.inference_mode()
def evaluate_perplexity(model, input_ids, context_length, stride, drop_remainder, description="Evaluating"):
    if stride != context_length:
        raise ValueError("Non-overlapping evaluation requires stride == context_length.")
    if not drop_remainder:
        raise ValueError("Paper-compatible evaluation requires drop_remainder=True.")
    if context_length > model.config.max_position_embeddings:
        raise ValueError("context_length exceeds the model context window.")

    device = next(model.parameters()).device
    sequence_length = input_ids.size(1)
    usable_length = sequence_length // context_length * context_length
    dropped_tokens = sequence_length - usable_length
    if usable_length == 0:
        raise ValueError("Input does not contain a complete context block.")

    total_nll = 0.0
    total_loss_tokens = 0
    total_blocks = usable_length // context_length

    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)
    start_time = time.perf_counter()

    for begin in tqdm(range(0, usable_length, stride), total=total_blocks, desc=description):
        end = begin + context_length
        batch = input_ids[:, begin:end].to(device)
        labels = batch.clone()

        loss = model(batch, labels=labels, use_cache=False).loss
        loss_tokens = labels[:, 1:].numel()
        total_nll += loss.float().item() * loss_tokens
        total_loss_tokens += loss_tokens

    torch.cuda.synchronize(device)
    elapsed_seconds = time.perf_counter() - start_time
    mean_nll = total_nll / total_loss_tokens

    return {
        "mean_nll": mean_nll,
        "perplexity": float(torch.exp(torch.tensor(mean_nll))),
        "source_input_tokens": sequence_length,
        "used_input_tokens": usable_length,
        "dropped_input_tokens": dropped_tokens,
        "evaluated_blocks": total_blocks,
        "evaluated_tokens": total_loss_tokens,
        "elapsed_seconds": elapsed_seconds,
        "tokens_per_second": total_loss_tokens / elapsed_seconds,
        "peak_gpu_memory_gib": torch.cuda.max_memory_allocated(device) / 2**30,
    }

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
full_evaluation = OB_SKIP_TEMPLATE.max_eval_tokens is None
results = []
output_paths = []

# ---------------------------------------------------------------------------
# DEWA-off baseline, measured in this run so it differs from the swept arms by
# exactly the DEWA on/off switch. The old BFP_REFERENCE_PPL_BY_MANTISSA table
# only covered mantissa_bits=7 and does not match this G/lm-head scope.
# ---------------------------------------------------------------------------
baseline_config = OBSkipConfig(
    threshold_bits=THRESHOLD_SWEEP[0],
    enabled=False,
    max_eval_tokens=OB_SKIP_TEMPLATE.max_eval_tokens,
)
configured_layers = configure_ob_skip(model, baseline_config, reset_stats=True)
assert configured_layers == len(quantized_layers), configured_layers
torch.cuda.empty_cache()

baseline_metrics = evaluate_perplexity(
    model,
    input_ids,
    CONTEXT_LENGTH,
    STRIDE,
    DROP_REMAINDER,
    description="Evaluating DEWA-off baseline",
)
baseline_aggregate, baseline_per_layer = collect_ob_skip_stats(model)
assert baseline_aggregate["skip_new"] == 0, baseline_aggregate["skip_new"]
assert baseline_aggregate["replace_old"] == 0, baseline_aggregate["replace_old"]
bfp_reference_ppl = baseline_metrics["perplexity"]

baseline_result = {
    "model": MODEL_ID,
    "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
    "split": SPLIT,
    "role": "DEWA-off baseline for the G16 control arm",
    "quantization": "W/A BFP fake quantization, DEWA disabled",
    "format": f"BFP{BFP_BITS} (1S{BFP.mantissa_bits}M + shared E{BFP.shared_exponent_bits})",
    "bfp_config": asdict(BFP),
    "ob_skip_config": asdict(baseline_config),
    "fake_bfp_storage_dtype": "float16",
    "group_partial_format": "float32",
    "accumulator_format": "float32 functional model",
    "linear_output_dtype": "float16",
    "matmul_backend": (
        f"Fused Triton Group-{BFP.block_size} dot with FP32 partials"
    ),
    "triton_kernel_config": asdict(KERNEL),
    "quantized_linear_layers": len(quantized_layers),
    "attention_implementation": "eager",
    "context_length": CONTEXT_LENGTH,
    "stride": STRIDE,
    "evaluation_protocol": EVALUATION_PROTOCOL,
    "drop_remainder": DROP_REMAINDER,
    "full_evaluation": full_evaluation,
    "fp16_reference_perplexity": FP16_BASELINE_PPL,
    "delta_perplexity_vs_fp16": (
        baseline_metrics["perplexity"] - FP16_BASELINE_PPL
        if full_evaluation and FP16_BASELINE_PPL is not None
        else None
    ),
    "ob_skip_stats": baseline_aggregate,
    "per_layer_ob_skip_stats": baseline_per_layer,
    "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda,
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "triton": triton.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    **baseline_metrics,
}
baseline_path = OUTPUT_DIR / (
    f"obskip-bfp{BFP_BITS}-g{BFP.block_size}-dewa-off-s2048.json"
)
baseline_path.write_text(
    json.dumps(baseline_result, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
output_paths.append(baseline_path)
print(
    f"Saved DEWA-off baseline: PPL={baseline_metrics['perplexity']:.6f} "
    f"-> {baseline_path}"
)

for threshold in THRESHOLD_SWEEP:
    ob_skip_config = OBSkipConfig(
        threshold_bits=threshold,
        enabled=OB_SKIP_TEMPLATE.enabled,
        max_eval_tokens=OB_SKIP_TEMPLATE.max_eval_tokens,
    )
    configured_layers = configure_ob_skip(model, ob_skip_config, reset_stats=True)
    assert configured_layers == len(quantized_layers), configured_layers
    torch.cuda.empty_cache()

    metrics = evaluate_perplexity(
        model,
        input_ids,
        CONTEXT_LENGTH,
        STRIDE,
        DROP_REMAINDER,
        description=f"Evaluating T={threshold}",
    )
    aggregate_stats, per_layer_stats = collect_ob_skip_stats(model)

    result = {
        "model": MODEL_ID,
        "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
        "split": SPLIT,
        "quantization": "W/A BFP fake quantization + fused Triton OB-Skip",
        "format": f"BFP{BFP_BITS} (1S{BFP.mantissa_bits}M + shared E{BFP.shared_exponent_bits})",
        "bfp_config": asdict(BFP),
        "ob_skip_config": asdict(ob_skip_config),
        "threshold_sweep": list(THRESHOLD_SWEEP),
        "fixed_bfp_model_reused_across_thresholds": True,
        "fake_bfp_storage_dtype": "float16",
        "group_partial_format": "float32",
        "accumulator_format": "float32 functional model",
        "linear_output_dtype": "float16",
        "matmul_backend": (
            f"Fused Triton Group-{BFP.block_size} dot with FP32 partials"
        ),
        "triton_kernel_config": asdict(KERNEL),
        "quantized_linear_layers": len(quantized_layers),
        "attention_implementation": "eager",
        "context_length": CONTEXT_LENGTH,
        "stride": STRIDE,
        "evaluation_protocol": EVALUATION_PROTOCOL,
        "drop_remainder": DROP_REMAINDER,
        "full_evaluation": full_evaluation,
        "evaluation_note": "Non-overlapping 2048-token blocks; incomplete final block is dropped.",
        "fp16_reference_perplexity": FP16_BASELINE_PPL,
        "same_bfp_reference_perplexity": bfp_reference_ppl,
        "delta_perplexity_vs_fp16": (
            metrics["perplexity"] - FP16_BASELINE_PPL
            if full_evaluation and FP16_BASELINE_PPL is not None
            else None
        ),
        "delta_perplexity_vs_same_bfp": (
            metrics["perplexity"] - bfp_reference_ppl
            if full_evaluation and bfp_reference_ppl is not None
            else None
        ),
        "ob_skip_stats": aggregate_stats,
        "per_layer_ob_skip_stats": per_layer_stats,
        "gpu": torch.cuda.get_device_name(0),
        "cuda": torch.version.cuda,
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "triton": triton.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        **metrics,
    }

    output_path = OUTPUT_DIR / (
        f"obskip-bfp{BFP_BITS}-g{BFP.block_size}-t{threshold}-s2048.json"
    )
    output_path.write_text(
        json.dumps(result, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    results.append(result)
    output_paths.append(output_path)
    print(
        f"Saved T={threshold}: PPL={metrics['perplexity']:.6f}, "
        f"OB-Skip rate={aggregate_stats['ob_skip_rate']:.6%} -> {output_path}"
    )

summary = [
    {
        "threshold_bits": result["ob_skip_config"]["threshold_bits"],
        "perplexity": result["perplexity"],
        "delta_vs_bfp": result["delta_perplexity_vs_same_bfp"],
        "ob_skip_rate": result["ob_skip_stats"]["ob_skip_rate"],
        "new_discard_rate": result["ob_skip_stats"]["new_discard_rate"],
        "old_replace_rate": result["ob_skip_stats"]["old_replace_rate"],
    }
    for result in results
]
print(json.dumps(summary, indent=2))

with zipfile.ZipFile(ARCHIVE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for output_path in output_paths:
        archive.write(output_path, arcname=output_path.name)

print(f"Archive: {ARCHIVE_PATH.resolve()}")

In [ ]:
from google.colab import files
files.download(str(ARCHIVE_PATH))